# 🍎 AI Fruit Classifier (Image + Weight)
Train with Transfer Learning (MobileNetV2)

In [ ]:
!pip install -q tensorflow pandas

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os
from tensorflow.keras import layers, models

## 📂 Upload dataset.zip + labels.csv

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
with zipfile.ZipFile("dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("dataset")

In [ ]:
IMG_SIZE = 224
df = pd.read_csv("labels.csv")

def load_data(df):
    images, weights, labels = [], [], []

    for _, row in df.iterrows():
        path = os.path.join("dataset", row['label'], row['filename'])

        img = tf.keras.preprocessing.image.load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
        img = tf.keras.preprocessing.image.img_to_array(img)
        img = tf.keras.applications.mobilenet_v2.preprocess_input(img)

        images.append(img)
        weights.append([row['weight']/300.0])
        labels.append(0 if row['label']=='apple' else 1)

    return np.array(images), np.array(weights), np.array(labels)

X_img, X_weight, y = load_data(df)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

for layer in base_model.layers:
    layer.trainable = False

In [ ]:
image_input = layers.Input(shape=(224,224,3))
x = base_model(image_input, training=False)
x = layers.GlobalAveragePooling2D()(x)

weight_input = layers.Input(shape=(1,))
w = layers.Dense(16, activation='relu')(weight_input)

combined = layers.concatenate([x,w])
z = layers.Dense(64, activation='relu')(combined)
z = layers.Dense(1, activation='sigmoid')(z)

model = models.Model(inputs=[image_input, weight_input], outputs=z)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit([X_img, X_weight], y, epochs=5, batch_size=8)

In [ ]:
model.save("fruit_model.h5")

In [ ]:
from google.colab import files
files.download("fruit_model.h5")